# Advanced packages - MAW (Multi-Aquifer Well)

Part of the **synthetic-valley advanced-packages** series, which upgrades a calibrated base model's simple boundary conditions to MODFLOW 6 *advanced packages* one notebook at a time. Each notebook loads the shared advanced model from `models/` (creating it from the base model on first use with `mf6_notebook_helpers.load_or_create_advanced_model`) and adds one package, so they can be run independently and in any order - except where a package depends on others.

- **UZF** (`mf6-advanced-packages-uzf`) - recharge (RCH) -> Unsaturated Zone Flow
- **MAW** (`mf6-advanced-packages-maw`) - wells (WEL) -> Multi-Aquifer Well *(this notebook)*
- **SFR** (`mf6-advanced-packages-sfr`) - river (RIV) -> Streamflow Routing
- **LAK** (`mf6-advanced-packages-lak`) - high-K lake -> Lake package
- **MVR** (`mf6-advanced-packages-mvr`) - Water Mover (requires UZF, LAK, SFR)
- **Processing output** (`mf6-advanced-packages-processing`) - run the model and plot results

Replace the simple well (WEL) package with the Multi-Aquifer Well (MAW) package, which represents wells screened across multiple layers and solves for a single head in each wellbore.

## Imports and setup

Import FloPy and the helpers, then choose the temporal resolution
(`sample_frequency`) and the model name.

In [ ]:
import flopy
from mf6_notebook_helpers import (
    drop_packages,
    load_or_create_advanced_model,
    load_temporal_data,
)

In [ ]:
sample_frequency = "annual"  # monthly or annual
name = "sv"

## Load or create the advanced model

Load the shared advanced model from `models/` (or create it from the base model on first use), then read the cell bottoms and the pumping time series.

In [ ]:
sim = load_or_create_advanced_model(sample_frequency, name)
gwf = sim.get_model(name)
nper = sim.tdis.nper.array
botm = gwf.dis.botm.array

In [ ]:
temporal_df = load_temporal_data(sample_frequency)

## Build the MAW package

Two multi-layer wells, each screened across model layers 4 and 5 (layer indices 3 and 4 in the zero-based cellids below). The wells are inactive during the (steady-state) calibration period and pump at the tabulated rate afterward.

In [ ]:
# well name -> (layer0, layer1, row, col)
well_data = {
    "reilly": (3, 4, 5, 14),
    "vc": (3, 4, 32, 5),
}

In [ ]:
# MAW package data
# <ifno> <radius> <bottom> <strt> <condeqn> <ngwfnodes> <boundname>
package_data = []
for idx, (key, value) in enumerate(well_data.items()):
    k0, k1, i, j = value
    package_data.append((idx, 0.5, float(botm[k1, i, j]), 0.0, "thiem", 2, key))

In [ ]:
# MAW connection data
# <ifno> <icon> <cellid> <scrn_top> <scrn_bot> <hk_skin> <radius_skin>
connection_data = []
for idx, (key, value) in enumerate(well_data.items()):
    k0, k1, i, j = value
    for jdx, k in enumerate(range(k0, k1 + 1)):
        top = float(botm[k - 1, i, j])
        bot = float(botm[k, i, j])
        connection_data.append((idx, jdx, (k, i, j), top, bot, -999.0, -999.0))

In [ ]:
# MAW stress period data: wells inactive in period 1, then active with a rate
maw_spd = {0: [(idx, "status", "inactive") for idx in range(len(well_data))]}
for n in range(1, nper):
    if n == 1:
        spd = [(idx, "status", "active") for idx in range(len(well_data))]
    else:
        spd = []
    row = temporal_df.iloc[n]
    for idx, well_name in enumerate(well_data.keys()):
        spd.append((idx, "rate", float(row[well_name])))
    maw_spd[n] = spd

In [ ]:
# replace the simple well package with MAW (reusing the "pwell" name)
drop_packages(gwf, "pwell")
maw = flopy.mf6.ModflowGwfmaw(
    gwf,
    boundnames=True,
    nmawwells=len(well_data),
    packagedata=package_data,
    connectiondata=connection_data,
    perioddata=maw_spd,
    pname="pwell",
)
gwf.get_package_list()

## Write and run

Write the updated advanced model back to `models/` and run it to confirm the MAW package is valid.

In [ ]:
sim.write_simulation(silent=True)
success, buff = sim.run_simulation(silent=True)
assert success, "MODFLOW 6 did not terminate normally"
print("MAW advanced model ran successfully")

## Recap

In this notebook you replaced the Well (**WEL**) package with the Multi-Aquifer Well (**MAW**) package, which represents one well screened across several layers and lets the model compute how much water it exchanges with each, then wrote the updated advanced model back to `models/` and ran it.